<a href="https://colab.research.google.com/github/RiazullJannat/DL/blob/main/backpropagation_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import torch

data = torch.tensor(
    [
        [2,3,3],
        [5,6,4],
        [7,8,5]

    ]
)
data

tensor([[2, 3, 3],
        [5, 6, 4],
        [7, 8, 5]])

In [58]:
x = data[:,0:2].float()
y = data[:,-1].float()

In [59]:
x

tensor([[2., 3.],
        [5., 6.],
        [7., 8.]])

In [60]:
y

tensor([3., 4., 5.])

**Weight Initialization**

In [61]:
# w=1, b = 0

def initialize_parameters():
  parameters = {}
  parameters["w1"] = torch.ones((2,2)) * 0.1
  parameters["b1"] = torch.ones((2,1))

  parameters["w2"] = torch.ones((2,1)) * 0.1
  parameters["b2"] = torch.zeros((1,1))

  return parameters

In [62]:
initialize_parameters()

{'w1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[1.],
         [1.]]),
 'w2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

**Batter way**

In [63]:
def initialize_parameters2(layer_dims):
  torch.manual_seed(3)
  parameters = {}
  L = len(layer_dims)

  for i in range(1,L):
    parameters["w"+str(i)] = torch.ones((layer_dims[i-1], layer_dims[i])) * 0.1
    parameters["b"+str(i)] = torch.zeros((layer_dims[i], 1))

  return parameters


In [64]:
initialize_parameters2([2,2,1])

{'w1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'w2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

**Forward Propagation**

In [65]:
def liner_forward(a_prev, w, b):
  z = torch.matmul(w.T, a_prev) + b
  return z

In [66]:
def l_layer_forward(x, parameters):
  a = x
  l = len(parameters)//2
  for i in range(1,l+1):
    a_prev = a

    wl = parameters['w' + str(i)]
    bl = parameters['b' + str(i)]

    a = liner_forward(a_prev, wl, bl)
  return a, a_prev

In [67]:
params = initialize_parameters2([2, 2, 1])
params

{'w1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'w2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

In [68]:
X_sample= x[0].reshape(2,1).float() # Convert X_sample to float
y_hat, a1 = l_layer_forward(X_sample, params)
print("y hat->> ", y_hat)
print(a1)

y hat->>  tensor([[0.1000]])
tensor([[0.5000],
        [0.5000]])


In [69]:
print("loss-->> ", (y - y_hat)**2)

loss-->>  tensor([[ 8.4100, 15.2100, 24.0100]])


In [70]:
def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):

    err_signal = 2 * (y - y_hat)

     #  Save OLD W2 values before updating
    W2_00_old = parameters['w2'][0, 0].item()
    W2_10_old = parameters['w2'][1, 0].item()

    # Update Layer 2
    parameters['w2'][0, 0] += lr * err_signal * A1[0, 0]
    parameters['w2'][1, 0] += lr * err_signal * A1[1, 0]
    parameters['b2'][0, 0] += lr * err_signal


    parameters['w1'][0, 0] += lr * err_signal * parameters['w2'][0, 0] * X[0, 0]
    parameters['w1'][0, 1] += lr * err_signal * parameters['w2'][0, 0] * X[1, 0]
    parameters['b1'][0, 0] += lr * err_signal * parameters['w2'][0, 0]

    parameters['w1'][1, 0] += lr * err_signal * parameters['w2'][1, 0] * X[0, 0]
    parameters['w1'][1, 1] += lr * err_signal * parameters['w2'][1, 0] * X[1, 0]
    parameters['b1'][1, 0] += lr * err_signal * parameters['w2'][1, 0]

    return parameters

params = initialize_parameters2([2, 2, 1])
epochs = 5

for i in range(epochs):
    epoch_loss = 0
    for j in range(x.shape[0]):
        # Prepare sample
        X_sample = x[j].reshape(2, 1)
        y_sample = y[j]

        # Forward
        y_hat, A1 = l_layer_forward(X_sample, params)
        y_hat_scalar = y_hat[0, 0]

        # Update
        params = update_parameters(params, y_sample, y_hat_scalar, A1, X_sample)

        # Loss calculation
        loss = (y_sample - y_hat_scalar) ** 2
        epoch_loss += loss.item()

    print(f"Epoch - {i+1} Loss - {epoch_loss / x.shape[0]}")

Epoch - 1 Loss - 14.67866039276123
Epoch - 2 Loss - 13.701451301574707
Epoch - 3 Loss - 12.494118690490723
Epoch - 4 Loss - 11.040447076161703
Epoch - 5 Loss - 9.366330782572428
